# 12 — Per-Compound Adaptive Blending

Global blend weights (e.g. 50% Chemprop / 50% LGBM) ignore the fact that the test set is heterogeneous:
- **High-similarity compounds** (top-1 Tanimoto > 0.6): kNN and MMP are most reliable
- **Medium-similarity** (0.4–0.6): LGBM ensemble is best
- **Scaffold hops** (< 0.4): Chemprop generalises best

This notebook computes per-compound blend weights via a sigmoid function of top-1 Tanimoto similarity,
optionally modulated by the test difficulty score from notebook 01.

**Runtime**: ~5 min (just fingerprint computation + loading saved predictions).

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

from pxr.data import load_train, load_test
from pxr.chem import morgan_fp_batch
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

plt.rcParams.update({"figure.dpi": 120})
print("Setup complete.")

In [ ]:
# ── 2. Load test predictions from all prior submissions ───────────────────────
train = load_train()
te    = load_test()

def load_sub(fname):
    p = SUBMISSIONS / fname
    if not p.exists():
        return None
    return pd.read_csv(p).set_index('Molecule Name').loc[te['name'].values, 'pEC50'].values

preds = {}
for label, fname in [
    ('lgbm_aug',    '07_final_ensemble.csv'),   # LGBM_aug+kNN component
    ('lgbm_pipe',   '09_lgbm_pipeline.csv'),
    ('meta',        '11_stacked_ensemble.csv'),
    ('chemprop',    '10_expanded_multitask.csv'),
    ('chemprop_08', '08_chemprop_cv_blend.csv'),
    ('knn',         '05_knn_blend.csv'),
]:
    p = load_sub(fname)
    if p is not None:
        preds[label] = p
        print(f"  {label:15s}: {p.min():.2f}–{p.max():.2f}  (median {np.median(p):.3f})")
    else:
        print(f"  {label:15s}: NOT FOUND — skipping")

In [ ]:
# ── 3. Compute test top-1 Tanimoto similarity ─────────────────────────────────
print("Computing ECFP4 fingerprints ...")
X_tr_fp = morgan_fp_batch(train['smiles'].tolist()).astype(bool)
X_te_fp = morgan_fp_batch(te['smiles'].tolist()).astype(bool)

nn = NearestNeighbors(n_neighbors=1, metric='jaccard', algorithm='brute', n_jobs=4)
nn.fit(X_tr_fp)
dists, _ = nn.kneighbors(X_te_fp)
top1_sim  = 1.0 - dists[:, 0]

print(f"Top-1 Tanimoto: mean={top1_sim.mean():.3f}  median={np.median(top1_sim):.3f}  "
      f"min={top1_sim.min():.3f}  max={top1_sim.max():.3f}")
print(f"sim > 0.6: {(top1_sim > 0.6).sum()}  |  sim > 0.4: {(top1_sim > 0.4).sum()}  "
      f"|  sim < 0.4: {(top1_sim < 0.4).sum()}")

In [ ]:
# ── 4. Load test difficulty score (from nb 01) ────────────────────────────────
diff_path = DATA_PROCESSED / 'test_difficulty.parquet'
if diff_path.exists():
    diff_df   = pd.read_parquet(diff_path)
    diff_df   = diff_df.set_index('name') if 'name' in diff_df.columns else diff_df
    diff_vals = diff_df.reindex(te['name'].values)['difficulty'].values
    # Normalise to [0, 1]
    nanrange  = np.nanmax(diff_vals) - np.nanmin(diff_vals) + 1e-9
    diff_norm = (diff_vals - np.nanmin(diff_vals)) / nanrange
    print(f"Test difficulty: {diff_norm.min():.3f}–{diff_norm.max():.3f}")
    use_difficulty = True
else:
    use_difficulty = False
    print("test_difficulty.parquet not found — using similarity only")

In [ ]:
# ── 5. Per-compound blend weights via sigmoid of similarity ──────────────────
def sigmoid(x, center=0.5, slope=10.0):
    return 1.0 / (1.0 + np.exp(-slope * (x - center)))

# kNN weight: high for high-sim compounds
# Centre at 0.55 (median sim is 0.52 — we want kNN to dominate only for clearly close analogs)
w_knn = sigmoid(top1_sim, center=0.55, slope=12.0) * 0.35   # max 35%

# Chemprop weight: high for low-sim (scaffold hops)
w_cp  = sigmoid(1.0 - top1_sim, center=0.55, slope=12.0) * 0.45   # max 45%

# LGBM gets the remainder
w_lgbm = np.clip(1.0 - w_knn - w_cp, 0.15, 0.65)

# Re-normalise to sum to 1
total  = w_knn + w_cp + w_lgbm
w_knn  /= total
w_cp   /= total
w_lgbm /= total

print(f"Weight ranges (across 513 test compounds):")
print(f"  kNN:     {w_knn.min():.3f} – {w_knn.max():.3f}   mean={w_knn.mean():.3f}")
print(f"  LGBM:    {w_lgbm.min():.3f} – {w_lgbm.max():.3f}  mean={w_lgbm.mean():.3f}")
print(f"  Chemprop:{w_cp.min():.3f} – {w_cp.max():.3f}  mean={w_cp.mean():.3f}")

In [ ]:
# ── 6. Assemble adaptive ensemble ─────────────────────────────────────────────
# Best available Chemprop and LGBM predictions
cp_preds   = preds.get('chemprop',    preds.get('chemprop_08'))
lgbm_preds = preds.get('meta',        preds.get('lgbm_pipe', preds.get('lgbm_aug')))
knn_preds  = preds.get('knn')

if knn_preds is None:
    # Fall back: weight kNN = 0, redistribute to LGBM + CP
    knn_preds = lgbm_preds
    print("kNN predictions not found — using LGBM as kNN fallback")

train_y = load_train()['pec50'].values
adaptive_preds = w_knn * knn_preds + w_lgbm * lgbm_preds + w_cp * cp_preds
adaptive_preds = np.clip(adaptive_preds, train_y.min() - 0.5, train_y.max() + 0.5)

print(f"Adaptive preds: {adaptive_preds.min():.2f} – {adaptive_preds.max():.2f}  "
      f"(median {np.median(adaptive_preds):.3f})")

In [ ]:
# ── 7. Visualise weight distribution ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Weight vs similarity
sort_idx = np.argsort(top1_sim)
ax = axes[0]
ax.plot(top1_sim[sort_idx], w_knn[sort_idx],  label='kNN',     color='steelblue')
ax.plot(top1_sim[sort_idx], w_lgbm[sort_idx], label='LGBM',    color='darkorange')
ax.plot(top1_sim[sort_idx], w_cp[sort_idx],   label='Chemprop',color='forestgreen')
ax.set(xlabel='Top-1 Tanimoto similarity', ylabel='Blend weight',
       title='Per-compound weights vs similarity')
ax.legend()

# Similarity histogram
axes[1].hist(top1_sim, bins=30, color='steelblue', edgecolor='k', lw=0.4)
axes[1].axvline(0.55, color='red', lw=1.5, linestyle='--', label='centre=0.55')
axes[1].set(xlabel='Top-1 Tanimoto', title='Test similarity distribution')
axes[1].legend()

# Prediction distribution
axes[2].hist(adaptive_preds, bins=30, color='forestgreen', edgecolor='k', lw=0.4)
axes[2].set(xlabel='pEC50', title='Adaptive ensemble predictions')

plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'figures' / '12_adaptive_blend.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 8. Save submission ───────────────────────────────────────────────────────
sub = pd.DataFrame({'Molecule Name': te['name'].values,
                    'SMILES':        te['smiles'].values,
                    'pEC50':         adaptive_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '12_adaptive_blend.csv'
sub.to_csv(out, index=False)
print(f"Saved: {out}")
print(f"Mean weights  kNN={w_knn.mean():.3f}  LGBM={w_lgbm.mean():.3f}  CP={w_cp.mean():.3f}")
print(sub['pEC50'].describe().round(3))